# Name That LLM: Standalone Inference & Submission Notebook
This notebook runs offline inside the Kaggle execution container, trains an ensembled TF-IDF + Stylometry baseline on `train.csv`, predicts on the hidden `test.csv`, and writes the required `submission.parquet`.

In [ ]:
import os
import re
import pathlib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import StandardScaler

print('Environment initialized successfully.')

In [ ]:
# 1. Locate Competition Data Paths
POSSIBLE_DIRS = [
    pathlib.Path('/kaggle/input/name-that-llm'),
    pathlib.Path('/kaggle/input/competitions/name-that-llm'),
    pathlib.Path('./data/raw'),
    pathlib.Path('./data/sample'),
]

DATA_DIR = None
for p in POSSIBLE_DIRS:
    if (p / 'train.csv').exists():
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train.csv in any expected path!')

print(f'Loading data from: {DATA_DIR}')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train shape: {train_df.shape} | Test shape: {test_df.shape}')

In [ ]:
# 2. Stylometric Feature Extractor
class StylometricExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, texts):
        rows = []
        for text in texts:
            s = str(text)
            chars = max(len(s), 1)
            words = re.findall(r'\b\w+\b', s)
            w_count = max(len(words), 1)
            sentences = [sent.strip() for sent in re.split(r'[.!?]+', s) if sent.strip()]
            s_count = max(len(sentences), 1)
            rows.append({
                'char_count': chars,
                'word_count': w_count,
                'avg_word_len': np.mean([len(w) for w in words]) if words else 0,
                'avg_sent_len': w_count / s_count,
                'semicolons': (s.count(';') / chars) * 1000,
                'colons': (s.count(':') / chars) * 1000,
                'commas': (s.count(',') / chars) * 1000,
                'em_dashes': ((s.count('—') + s.count('–')) / chars) * 1000,
                'quotes': ((s.count('"') + s.count("'") + s.count('“') + s.count('”')) / chars) * 1000,
                'bullets': len(re.findall(r'(?:^|\n)\s*[-*•]\s+', s)),
                'numbers': len(re.findall(r'(?:^|\n)\s*\d+\.\s+', s)),
                'bold': len(re.findall(r'\*\*[^*]+\*\*', s)),
                'code': len(re.findall(r'`[^`]+`', s)),
                'uppercase_ratio': sum(1 for c in s if c.isupper()) / chars,
                'ttr': len(set(w.lower() for w in words)) / w_count,
            })
        return pd.DataFrame(rows).values

In [ ]:
# 3. Model Pipeline
TARGET_MODELS = sorted(train_df['model'].unique().tolist())
print(f'Total target classes ({len(TARGET_MODELS)}): {TARGET_MODELS}')

tfidf = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=25000, sublinear_tf=True, min_df=2)),
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=35000, sublinear_tf=True, min_df=2)),
])
sty = StylometricExtractor()
scaler = StandardScaler()

X_texts = train_df['text'].fillna('').values
y = train_df['model'].values
groups = train_df['prompt_id'].values

print('Extracting TF-IDF and Stylometric features...')
X_tfidf = tfidf.fit_transform(X_texts)
X_sty = scaler.fit_transform(sty.fit_transform(X_texts))
X_train = sp.hstack([X_tfidf, sp.csr_matrix(X_sty)])

# 5-fold StratifiedGroupKFold Ensembling
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
models = []
for fold, (t_idx, v_idx) in enumerate(sgkf.split(X_train, y, groups)):
    clf = LogisticRegression(C=2.0, max_iter=1500, class_weight='balanced', random_state=42)
    clf.fit(X_train.tocsr()[t_idx], y[t_idx])
    val_acc = (clf.predict(X_train.tocsr()[v_idx]) == y[v_idx]).mean()
    print(f'Fold {fold + 1} validation accuracy: {val_acc:.4f}')
    models.append(clf)

In [ ]:
# 4. Inference on Test Set
test_texts = test_df['text'].fillna('').values
X_test_tfidf = tfidf.transform(test_texts)
X_test_sty = scaler.transform(sty.transform(test_texts))
X_test = sp.hstack([X_test_tfidf, sp.csr_matrix(X_test_sty)])

# Average probabilities across fold models
classes = models[0].classes_
test_probs = np.zeros((len(test_df), len(classes)))
for clf in models:
    test_probs += clf.predict_proba(X_test)
test_probs /= len(models)

final_preds = classes[np.argmax(test_probs, axis=1)]

# 5. Format & Validate Submission
submission = pd.DataFrame({
    'id': test_df['id'],
    'model': final_preds
})

assert list(submission.columns) == ['id', 'model'], 'Invalid submission columns!'
assert not submission.isna().any().any(), 'Submission contains NaNs!'
assert len(submission) == len(test_df), 'Submission row count mismatch!'

# Output official submission.parquet
submission.to_parquet('submission.parquet', index=False)
print('✓ Successfully saved submission.parquet!')
print(submission.head(10))